# Config stuff

In [2]:

import ConnectionConfig as cc
from delta import DeltaTable
from datetime import datetime

In [3]:
cc.setupEnvironment()
spark = cc.startLocalCluster("dimUserIncrementalLoad")
spark.getActiveSession()

## Incremental load

After the sales Rep dimension is filled for the first time, the logic to update the dimension has to be handled differently. A change of a record in the source system has to be handled as a change in the dimension. The SCD2 logic is used to handle this.

The SCD2 implementation requires a more complex transformation to correctly handle changes in the source files. For detailed information consult the comments in the code.
### Setting the parameters
The timestamp of the job is used to set the scd_end date of the previous record and the scd_start date of the new record.

In [4]:
run_timestamp =datetime.now() #The job runtime is stored in a variable

### Read existing dimension

In [5]:
dt_dimUser = DeltaTable.forPath(spark,".\\spark-warehouse\\dimuser")

dt_dimUser.toDF().createOrReplaceTempView("dimUsers_current")

#DEBUG CODE TO SHOW CONTENT OF DIMENSION
spark.sql("select userid, name, street, SK, md5  from dimUsers_current").show()

+------+--------------------+--------------------+--------------------+--------------------+
|userid|                name|              street|                  SK|                 md5|
+------+--------------------+--------------------+--------------------+--------------------+
|    15|           Hoek Emma|    John Kennedylaan|d0fed1b7-2186-48e...|6e3779642d13bf4ab...|
|    16|     Stevens Suzanne|              Wijk 2|99963c5d-a30d-4de...|81dec6ab4f1b91eef...|
|    17|van den Bosch Rutger|       Schoggestraat|bfbb7bb2-22f1-40d...|f4b825db3b59e9970...|
|    18|      Veldman Angela|Emiel Van Hemeldo...|bccc1d7a-2e7a-4e6...|319ef7ee26c646ab2...|
|    19|     Bakker Daniëlle|            Fortbaan|cd2bc5bc-5671-436...|ccc59a80114fbf09b...|
|    20|      Vink Elisabeth|  Joseph Wateletlaan|a21dc9d7-581b-443...|aa838917117b89e28...|
|    21|        Muller Simon|    Montevideostraat|9118695f-58c9-440...|12079fbd7516de0ef...|
|    22|     Dijkstra Justin|Joseph Van Hellem...|1a550016-ae38-4f3...

### Read source table


##### 1 READ SOURCE TABLE
Creating dataframe with source table (from operational system). Transformed to the dimension format.
The surrogate key is a uuid to be sure it's unique.
md5 hash is used to identify changes in the source table.
A view is created of the resulting dataframe to make it available for the next step.

In [6]:
cc.set_connectionProfile("velodb")

#a. Reading from a JDBC source
df_operational_sales_rep = spark.read \
    .format("jdbc") \
    .option("driver" , cc.get_Property("driver")) \
    .option("url", cc.create_jdbc()) \
    .option("dbtable", "velo_users") \
    .option("user", cc.get_Property("username")) \
    .option("password", cc.get_Property("password")) \
    .option("partitionColumn", "userid") \
    .option("numPartitions", 4) \
    .option("lowerBound", 0) \
    .option("upperBound", 20) \
    .load()

df_operational_sales_rep.createOrReplaceTempView("operational_users")

#b. Transforming the source to the dimension format
df_operational_users_new = spark.sql( "select uuid() as userSK, \
                                        userid as source_userId, \
                                        name as source_name, \
                                        street as source_street, \
                                      number as source_number, \
                                      zipcode as source_zipcode, \
                                      city as source_city, \
                                      country_code as source_country_code, \
                                        md5(concat(street, number, zipcode, city, country_code)) as source_md5 \
                                    from operational_users")

df_operational_users_new.createOrReplaceTempView("dimUsers_new")

#DEBUG CODE TO SHOW CONTENT OF SOURCE
#df_dim_sales_rep_new.printSchema()
#df_dim_sales_rep_new.show()
spark.sql("select * from dimUsers_new").show()
#df_dim_sales_rep.write.format("delta").mode("overwrite").saveAsTable("dimSalesRep")


+--------------------+-------------+--------------------+--------------------+-------------+--------------+--------------------+-------------------+--------------------+
|              userSK|source_userId|         source_name|       source_street|source_number|source_zipcode|         source_city|source_country_code|          source_md5|
+--------------------+-------------+--------------------+--------------------+-------------+--------------+--------------------+-------------------+--------------------+
|65abbb6f-2187-470...|            1|         Bouman Lars|         Somméstraat|         156 |          2060|           Antwerpen|                 BE|fddd2c34acf178b1f...|
|130e4623-2ca5-414...|            2|   van der Zee Julia|          Europalaan|          43 |          2610| Wilrijk (Antwerpen)|                 BE|e91f35aa29e5d732f...|
|22a71c17-e957-40c...|            3|     de Boer Ricardo|   Maria Clarastraat|          80 |          2160|           Wommelgem|                 BE|53


##### 2 DETECT CHANGES
Dataframe to identify SCD2 changed rows.
First a join between SOURCE (operational system) and DIMENSION (dwh) is performed
   The md5 hash is used to identify differences.
   The list contains:
       - updated source-rows (the join finds a rowand the md5 is different)  and
       - new source-rows (the leftouter join does not find a row in the dimension (dwh.salesRepId is null)

In [7]:

detectedChanges=spark.sql(f"select * \
                          from dimUsers_new source \
                          left outer join dimUsers_current dwh on dwh.userid == source.source_userId and dwh.current == true \
                          where dwh.userid is null or dwh.md5 <> source.source_md5")

detectedChanges.createOrReplaceTempView("detectedChanges")

#DEBUG CODE TO SHOW CONTENT OF DETECTED CHANGES
detectedChanges.show()


+------+-------------+-----------+-------------+-------------+--------------+-----------+-------------------+----------+---+------+----+-----+------+------+-------+----+------------+---------+-------+---+-------+
|userSK|source_userId|source_name|source_street|source_number|source_zipcode|source_city|source_country_code|source_md5| SK|userid|name|email|street|number|zipcode|city|country_code|scd_start|scd_end|md5|current|
+------+-------------+-----------+-------------+-------------+--------------+-----------+-------------------+----------+---+------+----+-----+------+------+-------+----+------------+---------+-------+---+-------+
+------+-------------+-----------+-------------+-------------+--------------+-----------+-------------------+----------+---+------+----+-----+------+------+-------+----+------------+---------+-------+---+-------+




##### 3 TRANSOFRM TO UPSERTS
Before union: Every updated and new source-row requires the insertion of a new record in the SCD2 dimension. This new records starts at the runtime of the job and ends at the end of time (2100-12-12). Current is set to true.
Updated source-rows also require an update of the existing scd-fields. The scd_end date of the existing record is set to the runtime of the job. Current is set to false

In the next step, rows without mergeKey will be inserted in the dimension table and rows with mergekey will be updated in the dimension

In [8]:

df_upserts = spark.sql(f"select SK as userSK,\
                                source_userId as userid,\
                                source_name as name,\
                                source_street as street,\
                                source_number as source_number, \
                                source_zipcode as zipcode, \
                                source_city as city, \
                                source_country_code as country_code, \
                                to_timestamp('{run_timestamp}') as scd_start, \
                                to_timestamp('2100-12-12','yyyy-MM-dd') as scd_end,\
                                source_md5 as md5,\
                                true as current\
                        from  detectedChanges\
                        union \
                        select  userSK,\
                                userId,\
                                name,\
                                street,\
                                number,\
                                zipcode,\
                                city, \
                                country_code, \
                                scd_start, \
                                to_timestamp('{run_timestamp}') as scd_end,\
                                md5, \
                                false \
                                from detectedChanges \
                        where current is not null")

df_upserts.createOrReplaceTempView("upserts")

In [9]:

#DEBUG CODE TO SHOW CONTENT OF UPSERTS
spark.sql("select * from upserts").show()

+------+------+----+------+-------------+-------+----+------------+---------+-------+---+-------+
|userSK|userid|name|street|source_number|zipcode|city|country_code|scd_start|scd_end|md5|current|
+------+------+----+------+-------------+-------+----+------------+---------+-------+---+-------+
+------+------+----+------+-------------+-------+----+------------+---------+-------+---+-------+




#### PERFORM MERGE DIMSALESREP AND UPSERTS
merge looks for a matching dwh.salesRepID (in the dimension) for mergeKey
   - when a match is found (the dimension table contains a row where its salesRepId corresponds with one of the mergekeys)  -> perform update of row to close the period and set current to "false"
   - when no match is found (there is no salesRepID in the dimension because the mergeKey is null) -> perform an insert with the data from the updserts table (from the source). The scd-start is filled with the run_timestamp)

In [10]:
spark.sql("""
MERGE INTO dimUsers_current AS target
USING upserts AS source
ON target.userid = source.userid
AND source.current = false
AND target.current = true
WHEN MATCHED THEN
    UPDATE SET
        target.scd_end = source.scd_end,
        target.current = source.current
WHEN NOT MATCHED THEN
    INSERT (SK, userid, name, street, scd_start, scd_end, md5, current)
    VALUES (source.userSK, source.userid, source.name, source.street, source.scd_start, source.scd_end, source.md5, source.current)
""")

#DEBUG CODE TO SHOW CONTENT OF DIMENSION
dt_dimUser.toDF().sort("userid", "scd_start").show(500)


+--------------------+------+--------------------+--------------------+--------------------+--------+-------+--------------------+------------+-------------------+-------------------+--------------------+-------+
|                  SK|userid|                name|               email|              street|  number|zipcode|                city|country_code|          scd_start|            scd_end|                 md5|current|
+--------------------+------+--------------------+--------------------+--------------------+--------+-------+--------------------+------------+-------------------+-------------------+--------------------+-------+
|f12cc15d-640e-49d...|     1|         Bouman Lars|Lars.Bouman@gmail...|         Somméstraat|    156 |   2060|           Antwerpen|          BE|1999-01-01 00:00:00|2100-12-12 00:00:00|fddd2c34acf178b1f...|   true|
|528425b9-9ace-43c...|     2|   van der Zee Julia|Julia.van.der.Zee...|          Europalaan|     43 |   2610| Wilrijk (Antwerpen)|          BE|1999-

## Delete the spark session

In [11]:
spark.stop()